In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from pathlib import Path
import time
from tqdm import tqdm

# ====================== CONFIGURACIÓN ======================
TARGET_SIZE = 224
CROP_ZOOM = 0.55
MIN_AREA_RATIO = 0.12  # Reducido para manos inclinadas
START_SEC = 3.0
FRAMES_PER_VIDEO = 3

TEMP_DIR = Path(r"C:\Users\Usuario\Desktop\UNIR\DATASETS\TEMP_VIDEOS")

# ====================== NUEVA CONFIGURACIÓN PARA CARPETAS ======================
OUTPUT_BASE = Path(r"C:\Users\Usuario\Desktop\UNIR\DATASETS\PROCESSED")  # Cambia si quieres otra ruta

# Las 3 categorías
CATEGORIES = ["normal", "leve", "moderada"]
# ====================== FUNCIÓN MEJORADA ======================

def improved_palm_crop_adaptive(frame, debug=False):
    """
    Versión mejorada que maneja:
    - Manos inclinadas
    - Manos muy cerca
    - Fragmentación de máscara
    - Partes oscuras
    """
    if frame is None or frame.size == 0:
        return None, {}
    
    h, w = frame.shape[:2]
    original = frame.copy()
    steps = {}
    
    # PASO 1: Frame original
    steps['original'] = original
    
    # PASO 2: Mejorar iluminación (para partes oscuras)
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))  # Más agresivo
    l_enhanced = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l_enhanced, a, b]), cv2.COLOR_LAB2BGR)
    steps['enhanced'] = enhanced
    
    # PASO 3: Múltiples espacios de color para mejor detección
    ycbcr = cv2.cvtColor(enhanced, cv2.COLOR_BGR2YCrCb)
    hsv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2HSV)
    
    # PASO 4: Threshold Triangle en Y (más robusto)
    y_channel = cv2.medianBlur(ycbcr[:, :, 0], 5)
    _, thresh_y = cv2.threshold(y_channel, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_TRIANGLE)
    steps['threshold_y'] = thresh_y
    
    # PASO 5: Rango de piel más amplio (para manos inclinadas)
    # PASO 5: Rango de piel MEJORADO para manos MUY PÁLIDAS / blancas
    hsv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2HSV)
    
    # Rango 1: Piel normal
    lower_skin1 = np.array([0, 20, 70], dtype=np.uint8)
    upper_skin1 = np.array([25, 255, 255], dtype=np.uint8)
    skin_mask1 = cv2.inRange(hsv, lower_skin1, upper_skin1)
    
    # Rango 2: Piel muy pálida (baja saturación)
    lower_skin2 = np.array([0, 8, 90], dtype=np.uint8)
    upper_skin2 = np.array([35, 255, 255], dtype=np.uint8)
    skin_mask2 = cv2.inRange(hsv, lower_skin2, upper_skin2)
    
    # Rango 3: Piel extremadamente clara / muy iluminada
    lower_skin3 = np.array([0, 3, 130], dtype=np.uint8)
    upper_skin3 = np.array([40, 180, 255], dtype=np.uint8)
    skin_mask3 = cv2.inRange(hsv, lower_skin3, upper_skin3)
    
    # Combinar todo
    skin_mask = cv2.bitwise_or(skin_mask1, skin_mask2)
    skin_mask = cv2.bitwise_or(skin_mask, skin_mask3)
    
    # Filtro adicional: detectar zonas brillantes (ayuda mucho con manos pálidas)
    gray = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)
    _, bright_mask = cv2.threshold(gray, 160, 255, cv2.THRESH_BINARY)
    skin_mask = cv2.bitwise_or(skin_mask, bright_mask)
    
    # Suavizado
    skin_mask = cv2.medianBlur(skin_mask, 7)
    skin_mask = cv2.GaussianBlur(skin_mask, (5, 5), 0)
    steps['skin_mask'] = skin_mask
    
    # PASO 6: Combinación adaptativa
    thresh = cv2.bitwise_and(thresh_y, skin_mask)
    steps['combined_mask'] = thresh
    
    # PASO 7: Morfología más agresiva para unir fragmentos
    kernel_close = np.ones((11, 11), np.uint8)  # Más grande para unir
    kernel_open = np.ones((5, 5), np.uint8)
    
    # Cerrar huecos (une fragmentos)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel_close, iterations=3)
    # Abrir para eliminar ruido pequeño
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_open, iterations=2)
    # Dilatar para expandir bordes
    kernel_dilate = np.ones((7, 7), np.uint8)
    thresh = cv2.dilate(thresh, kernel_dilate, iterations=2)
    steps['morphology'] = thresh
    
    # PASO 8: Encontrar contornos con detección de manos inclinadas
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        # Si no hay contornos, usar umbralización en otro canal
        return fallback_crop_advanced(original), steps
    
    # Filtrar contornos por área y forma
    valid_contours = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > (h * w * 0.05):  # Mínimo 5% del frame
            # Verificar que sea una forma convexa (mano)
            hull = cv2.convexHull(cnt)
            hull_area = cv2.contourArea(hull)
            if hull_area > 0:
                convexity_ratio = area / hull_area
                if convexity_ratio > 0.3:  # Manos tienen convexidad alta
                    valid_contours.append(cnt)
    
    if not valid_contours:
        # Si no hay contornos válidos, usar el más grande
        largest = max(contours, key=cv2.contourArea)
    else:
        # Usar el contorno con mayor área entre los válidos
        largest = max(valid_contours, key=cv2.contourArea)
    
    area_ratio = cv2.contourArea(largest) / (h * w)
    
    if area_ratio < MIN_AREA_RATIO:
        # Si el área es muy pequeña, intentar con otro método
        return fallback_crop_advanced(original), steps
    
    # PASO 9: Crear máscara final más suave
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(mask, [largest], -1, 255, -1)
    
    # Suavizar bordes de la máscara
    mask = cv2.GaussianBlur(mask, (15, 15), 0)
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    # Rellenar huecos internos
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8), iterations=3)
    steps['final_mask'] = mask
    
    # PASO 10: Overlay mejorado
    overlay = original.copy()
    overlay[mask == 255] = overlay[mask == 255] * 0.6 + np.array([0, 255, 0]) * 0.4
    steps['mask_overlay'] = overlay.astype(np.uint8)
    
      # PASO 11: Centro mejorado (usando momentos + percentil)
    pts = np.where(mask == 255)
    if len(pts[0]) == 0:
        return fallback_crop_advanced(original), steps

    # === NUEVO CÁLCULO DE CENTRO MÁS ROBUSTO ===
    # 1. Usar momentos del contorno (mejor para forma irregular de mano)
    M = cv2.moments(largest)
    if M["m00"] != 0:
        center_x = int(M["m10"] / M["m00"])
        center_y = int(M["m01"] / M["m00"])
    else:
        # Fallback al método anterior
        center_y = int(np.percentile(pts[0], 50))
        center_x = int(np.mean(pts[1]))

    # 2. Ajuste fino según inclinación
    y_distribution = np.bincount(pts[0])
    if len(y_distribution) > 0:
        dense_y = np.argmax(y_distribution)
        # Si hay mucha diferencia, tirar un poco hacia la parte más densa (palma)
        if abs(dense_y - center_y) > h * 0.12:
            center_y = int((center_y * 0.7) + (dense_y * 0.3))

    # 3. Evitar que el centro quede muy arriba (común en manos abiertas)
    if center_y < h * 0.35:
        center_y = int(center_y * 1.1)  # bajar un poco el centro

    center_debug = original.copy()
    cv2.circle(center_debug, (center_x, center_y), 10, (0, 0, 255), -1)
    cv2.circle(center_debug, (center_x, center_y), 25, (0, 255, 0), 3)
    steps['center_detected'] = center_debug
    
    # PASO 12: Recorte adaptativo
    # PASO 12: Recorte MÁS CENTRADO Y AJUSTADO solo en la palma (estilo Gahan)
    x, y, bw, bh = cv2.boundingRect(largest)
    
    # === NUEVO RECORTE MÁS ESTRECHO Y CENTRADO EN PALMA ===
    # Usamos el centro calculado por momentos (más preciso)
    M = cv2.moments(largest)
    if M["m00"] != 0:
        center_x = int(M["m10"] / M["m00"])
        center_y = int(M["m01"] / M["m00"])
    else:
        center_y = int(np.percentile(pts[0], 50))
        center_x = int(np.mean(pts[1]))

    # Ajuste para palma: bajamos un poco el centro (los dedos suelen estar arriba)
    center_y = int(center_y * 1.08)   # ← Ajusta este valor si quieres más/menos abajo

    # Recorte más pequeño y centrado (más agresivo que antes)
   # === AQUÍ ES DONDE CAMBIAS EL TAMAÑO DEL RECORTE ===
    crop_side = int(max(bw, bh) * 0.48)     # ← Baja este número (antes estaba en 0.62)
    
    # Mínimo tamaño razonable
    min_crop = int(min(h, w) * 0.42)        # ← También puedes bajar un poco esto
    crop_side = max(crop_side, min_crop)
    crop_side = max(crop_side, min_crop)

    start_x = max(0, center_x - crop_side // 2)
    start_y = max(0, center_y - crop_side // 2)
    end_x = min(w, center_x + crop_side // 2)
    end_y = min(h, center_y + crop_side // 2)

    crop_debug = original.copy()
    cv2.rectangle(crop_debug, (start_x, start_y), (end_x, end_y), (0, 255, 0), 3)
    steps['crop_box'] = crop_debug

    roi = frame[start_y:end_y, start_x:end_x]

    # Si el ROI es demasiado pequeño, fallback
    if roi.shape[0] < 100 or roi.shape[1] < 100:
        return fallback_crop_advanced(original), steps

    # PASO 13: Resultado final
    resized = cv2.resize(roi, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA)
    
    # Mejora de contraste final (como en artículos médicos)
    lab_final = cv2.cvtColor(resized, cv2.COLOR_BGR2LAB)
    l_final, a_final, b_final = cv2.split(lab_final)
    clahe_final = cv2.createCLAHE(clipLimit=2.8, tileGridSize=(8, 8))
    l_final = clahe_final.apply(l_final)
    final = cv2.cvtColor(cv2.merge([l_final, a_final, b_final]), cv2.COLOR_LAB2BGR)

    steps['final_result'] = final

    return final, steps

def fallback_crop_advanced(frame):
    """Recorte de respaldo más robusto"""
    h, w = frame.shape[:2]
    
    # Detectar la región más brillante (asumimos que es la mano)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (15, 15), 0)
    
    # Encontrar la región más brillante
    _, thresh = cv2.threshold(gray, np.mean(gray), 255, cv2.THRESH_BINARY)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(largest)
        
        # Expandir ligeramente el bounding box
        padding = int(min(bw, bh) * 0.1)
        x = max(0, x - padding)
        y = max(0, y - padding)
        bw = min(w - x, bw + 2*padding)
        bh = min(h - y, bh + 2*padding)
        
        roi = frame[y:y+bh, x:x+bw]
    else:
        # Último recurso: recorte central
        crop_size = int(min(h, w) * 0.6)
        start_y = (h - crop_size) // 2
        start_x = (w - crop_size) // 2
        roi = frame[start_y:start_y + crop_size, start_x:start_x + crop_size]
    
    if roi.size == 0:
        roi = frame
    
    # Redimensionar manteniendo proporción
    return cv2.resize(roi, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA)

def show_timeline_enhanced(images_dict, title, figsize=(20, 12)):
    """Muestra línea de tiempo con pasos mejorados"""
    # Agregar nuevos pasos al timeline
    step_order = [
        'original',
        'enhanced',  # Iluminación mejorada
        'threshold_y',
        'skin_mask',
        'combined_mask',
        'morphology',
        'mask_overlay',
        'center_detected',
        'crop_box',
        'final_result'
    ]
    
    step_names = {
        'original': '1. Original',
        'enhanced': '2. Iluminación Mejorada',
        'threshold_y': '3. Threshold Y',
        'skin_mask': '4. Máscara Piel (Amplia)',
        'combined_mask': '5. Máscara Combinada',
        'morphology': '6. Morfología (Unida)',
        'mask_overlay': '7. Máscara Final',
        'center_detected': '8. Centro Detectado',
        'crop_box': '9. Recorte',
        'final_result': '10. Resultado Final'
    }
    
    available_steps = [(step, images_dict[step]) for step in step_order 
                      if step in images_dict and images_dict[step] is not None]
    
    if not available_steps:
        print(f"⚠️ No hay pasos disponibles")
        return
    
    n_steps = len(available_steps)
    cols = min(5, n_steps)  # Más columnas para más pasos
    rows = (n_steps + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if n_steps == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
    
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    for i, (step_key, img) in enumerate(available_steps):
        if i < len(axes):
            if len(img.shape) == 3 and img.shape[2] == 3:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            else:
                img_rgb = img
                
            axes[i].imshow(img_rgb, cmap='gray' if len(img.shape) == 2 else None)
            axes[i].set_title(step_names.get(step_key, step_key), fontsize=9, fontweight='bold')
            axes[i].axis('off')
    
    for i in range(len(available_steps), len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

def extract_frames_from_video(video_path, num_frames=4, start_sec=3.0):
    """Extrae frames equiespaciados de un video"""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    start_frame = int(start_sec * fps)
    if start_frame >= total_frames:
        cap.release()
        return []
    
    available_frames = total_frames - start_frame
    if available_frames < num_frames:
        num_frames = max(1, available_frames)
    
    # Seleccionar frames con mejor visibilidad (usar más frames)
    frame_indices = np.linspace(start_frame, total_frames-1, num_frames, dtype=int)
    
    extracted_frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            timestamp = idx / fps
            extracted_frames.append({
                'frame': frame,
                'timestamp': timestamp,
                'frame_idx': idx
            })
    
    cap.release()
    return extracted_frames

def process_all_videos_enhanced():
    """Procesa videos por categorías, salta si ya existe la imagen"""
   
    if not TEMP_DIR.exists():
        print(f"❌ No existe la carpeta: {TEMP_DIR}")
        return

    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    print(f"📁 Guardando imágenes en: {OUTPUT_BASE}")
    print("=" * 80)

    success_count = 0
    skipped_count = 0
    total_videos = 0

    for category in CATEGORIES:
        category_dir = TEMP_DIR / category
        if not category_dir.exists():
            print(f"⚠️ Carpeta no encontrada: {category}")
            continue

        output_category = OUTPUT_BASE / category
        output_category.mkdir(parents=True, exist_ok=True)

        video_extensions = ['.mp4', '.MP4', '.avi', '.AVI', '.mov', '.MOV']
        videos = []
        for ext in video_extensions:
            videos.extend(category_dir.glob(f"*{ext}"))

        print(f"\n📂 Procesando categoría: {category.upper()} → {len(videos)} videos")
        
        for video_idx, video_path in enumerate(videos, 1):
            total_videos += 1
            video_id = video_path.stem  # Ej: ID762

            # === NUEVA LÓGICA: Verificar si ya existe la imagen ===
            output_path = output_category / f"{video_id}.jpg"
            
            if output_path.exists():
                print(f"⏭️  Ya existe imagen para {video_id} → Saltando video")
                skipped_count += 1
                continue

            print(f"\n🎬 [{category}] Video {video_idx}/{len(videos)}: {video_path.name}")
            
            # Extraer frame
            frames = extract_frames_from_video(video_path, num_frames=1, start_sec=START_SEC)
            
            if not frames:
                print(f"⚠️ No se pudo extraer frame de {video_path.name}")
                continue

            frame = frames[0]['frame']

            # Procesar
            final, steps = improved_palm_crop_adaptive(frame, debug=True)
            
            if final is not None:
                success = cv2.imwrite(str(output_path), final)
                
                if success:
                    print(f"✅ Guardado: {output_path.name} | Tamaño: {final.shape}")
                    
                    # Eliminar video solo si se guardó correctamente
                    try:
                        video_path.unlink()
                        print(f"🗑️  Video eliminado: {video_path.name}")
                    except Exception as e:
                        print(f"⚠️ No se pudo eliminar video: {e}")
                    
                    success_count += 1
                else:
                    print(f"⚠️ Error al guardar {video_id}")
            else:
                print(f"⚠️ Falló el procesamiento de {video_id}")

            time.sleep(0.2)

    print("\n" + "=" * 80)
    print(f"📊 RESUMEN FINAL:")
    print(f" ✅ Procesados correctamente : {success_count}")
    print(f" ⏭️  Saltados (ya existían)  : {skipped_count}")
    print(f" 📁 Total videos revisados  : {total_videos}")
    print(f" 📁 Imágenes guardadas en   : {OUTPUT_BASE}")
    print("=" * 80)


# ====================== EJECUCIÓN ======================

if __name__ == "__main__":
    process_all_videos_enhanced()

In [5]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
from tqdm import tqdm

# ====================== CONFIGURACIÓN ======================
TARGET_SIZE = 224
CROP_ZOOM = 0.55
MIN_AREA_RATIO = 0.12
START_SEC = 3.0

# Rutas (ajusta si es necesario)
TEMP_DIR = Path(r"C:\Users\Usuario\Desktop\UNIR\DATASETS\TEMP_VIDEOS")
DEBUG_BASE = Path(r"C:\Users\Usuario\Desktop\UNIR\DATASETS\DEBUG_PREPROCESS")

# Las 3 categorías
CATEGORIES = ["normal", "leve", "moderada"]

# ====================== TUS FUNCIONES ORIGINALES (copiadas) ======================
# (Se incluyen aquí para que el script sea autónomo)

def improved_palm_crop_adaptive(frame, debug=False):
    """Versión mejorada que maneja: manos inclinadas, pálidas, etc."""
    if frame is None or frame.size == 0:
        return None, {}
    
    h, w = frame.shape[:2]
    original = frame.copy()
    steps = {}
    
    steps['original'] = original
    
    # PASO 2: Mejorar iluminación
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l_enhanced, a, b]), cv2.COLOR_LAB2BGR)
    steps['enhanced'] = enhanced
    
    # PASO 3-4: Threshold y máscara de piel
    ycbcr = cv2.cvtColor(enhanced, cv2.COLOR_BGR2YCrCb)
    y_channel = cv2.medianBlur(ycbcr[:, :, 0], 5)
    _, thresh_y = cv2.threshold(y_channel, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_TRIANGLE)
    steps['threshold_y'] = thresh_y
    
    hsv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2HSV)
    
    # Rangos de piel ampliados
    lower_skin1 = np.array([0, 20, 70], dtype=np.uint8)
    upper_skin1 = np.array([25, 255, 255], dtype=np.uint8)
    skin_mask1 = cv2.inRange(hsv, lower_skin1, upper_skin1)
    
    lower_skin2 = np.array([0, 8, 90], dtype=np.uint8)
    upper_skin2 = np.array([35, 255, 255], dtype=np.uint8)
    skin_mask2 = cv2.inRange(hsv, lower_skin2, upper_skin2)
    
    lower_skin3 = np.array([0, 3, 130], dtype=np.uint8)
    upper_skin3 = np.array([40, 180, 255], dtype=np.uint8)
    skin_mask3 = cv2.inRange(hsv, lower_skin3, upper_skin3)
    
    skin_mask = cv2.bitwise_or(skin_mask1, skin_mask2)
    skin_mask = cv2.bitwise_or(skin_mask, skin_mask3)
    
    gray = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)
    _, bright_mask = cv2.threshold(gray, 160, 255, cv2.THRESH_BINARY)
    skin_mask = cv2.bitwise_or(skin_mask, bright_mask)
    
    skin_mask = cv2.medianBlur(skin_mask, 7)
    skin_mask = cv2.GaussianBlur(skin_mask, (5, 5), 0)
    steps['skin_mask'] = skin_mask
    
    # Combinación
    thresh = cv2.bitwise_and(thresh_y, skin_mask)
    steps['combined_mask'] = thresh
    
    # Morfología
    kernel_close = np.ones((11, 11), np.uint8)
    kernel_open = np.ones((5, 5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel_close, iterations=3)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_open, iterations=2)
    kernel_dilate = np.ones((7, 7), np.uint8)
    thresh = cv2.dilate(thresh, kernel_dilate, iterations=2)
    steps['morphology'] = thresh
    
    # Contornos
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return fallback_crop_advanced(original), steps
    
    valid_contours = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > (h * w * 0.05):
            hull = cv2.convexHull(cnt)
            hull_area = cv2.contourArea(hull)
            if hull_area > 0:
                convexity_ratio = area / hull_area
                if convexity_ratio > 0.3:
                    valid_contours.append(cnt)
    
    if not valid_contours:
        largest = max(contours, key=cv2.contourArea)
    else:
        largest = max(valid_contours, key=cv2.contourArea)
    
    area_ratio = cv2.contourArea(largest) / (h * w)
    if area_ratio < MIN_AREA_RATIO:
        return fallback_crop_advanced(original), steps
    
    # Máscara final
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(mask, [largest], -1, 255, -1)
    mask = cv2.GaussianBlur(mask, (15, 15), 0)
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8), iterations=3)
    steps['final_mask'] = mask  # aunque no siempre se usa en timeline
    
    # Overlay
    overlay = original.copy()
    overlay[mask == 255] = overlay[mask == 255] * 0.6 + np.array([0, 255, 0]) * 0.4
    steps['mask_overlay'] = overlay.astype(np.uint8)
    
    # Centro
    M = cv2.moments(largest)
    if M["m00"] != 0:
        center_x = int(M["m10"] / M["m00"])
        center_y = int(M["m01"] / M["m00"])
    else:
        pts = np.where(mask == 255)
        center_y = int(np.percentile(pts[0], 50))
        center_x = int(np.mean(pts[1]))
    
    center_y = int(center_y * 1.08)
    center_debug = original.copy()
    cv2.circle(center_debug, (center_x, center_y), 10, (0, 0, 255), -1)
    cv2.circle(center_debug, (center_x, center_y), 25, (0, 255, 0), 3)
    steps['center_detected'] = center_debug
    
    # Recorte
    x, y, bw, bh = cv2.boundingRect(largest)
    crop_side = int(max(bw, bh) * 0.48)
    min_crop = int(min(h, w) * 0.42)
    crop_side = max(crop_side, min_crop)
    
    start_x = max(0, center_x - crop_side // 2)
    start_y = max(0, center_y - crop_side // 2)
    end_x = min(w, center_x + crop_side // 2)
    end_y = min(h, center_y + crop_side // 2)
    
    crop_debug = original.copy()
    cv2.rectangle(crop_debug, (start_x, start_y), (end_x, end_y), (0, 255, 0), 3)
    steps['crop_box'] = crop_debug
    
    roi = frame[start_y:end_y, start_x:end_x]
    if roi.shape[0] < 100 or roi.shape[1] < 100:
        return fallback_crop_advanced(original), steps
    
    resized = cv2.resize(roi, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA)
    
    # Contraste final
    lab_final = cv2.cvtColor(resized, cv2.COLOR_BGR2LAB)
    l_final, a_final, b_final = cv2.split(lab_final)
    clahe_final = cv2.createCLAHE(clipLimit=2.8, tileGridSize=(8, 8))
    l_final = clahe_final.apply(l_final)
    final = cv2.cvtColor(cv2.merge([l_final, a_final, b_final]), cv2.COLOR_LAB2BGR)
    steps['final_result'] = final
    
    return final, steps

def fallback_crop_advanced(frame):
    h, w = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (15, 15), 0)
    _, thresh = cv2.threshold(gray, np.mean(gray), 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(largest)
        padding = int(min(bw, bh) * 0.1)
        x = max(0, x - padding)
        y = max(0, y - padding)
        bw = min(w - x, bw + 2*padding)
        bh = min(h - y, bh + 2*padding)
        roi = frame[y:y+bh, x:x+bw]
    else:
        crop_size = int(min(h, w) * 0.6)
        start_y = (h - crop_size) // 2
        start_x = (w - crop_size) // 2
        roi = frame[start_y:start_y + crop_size, start_x:start_x + crop_size]
    
    if roi.size == 0:
        roi = frame
    return cv2.resize(roi, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_AREA)

def show_timeline_enhanced(steps, video_id, save_path=None):
    """Genera timeline profesional para artículo"""
    step_order = [
        'original', 'enhanced', 'threshold_y', 'skin_mask',
        'combined_mask', 'morphology', 'mask_overlay',
        'center_detected', 'crop_box', 'final_result'
    ]
    
    step_names = {
        'original': '1. Frame Original',
        'enhanced': '2. Iluminación Mejorada (CLAHE)',
        'threshold_y': '3. Threshold en Canal Y',
        'skin_mask': '4. Máscara de Piel (Multi-rango)',
        'combined_mask': '5. Máscara Combinada',
        'morphology': '6. Morfología (Cierre + Dilatación)',
        'mask_overlay': '7. Overlay de Máscara',
        'center_detected': '8. Centro Detectado',
        'crop_box': '9. Bounding Box de Recorte',
        'final_result': '10. Resultado Final (224x224)'
    }
    
    available_steps = [(step, steps[step]) for step in step_order if step in steps and steps[step] is not None]
    
    n_steps = len(available_steps)
    cols = min(5, n_steps)
    rows = (n_steps + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(20, 12))
    if n_steps == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
    
    fig.suptitle(f'Pipeline de Preprocesamiento - Video: {video_id}', fontsize=18, fontweight='bold')
    
    for i, (step_key, img) in enumerate(available_steps):
        if i < len(axes):
            if len(img.shape) == 3 and img.shape[2] == 3:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            else:
                img_rgb = img
            axes[i].imshow(img_rgb, cmap='gray' if len(img.shape) == 2 else None)
            axes[i].set_title(step_names.get(step_key, step_key), fontsize=11, fontweight='bold')
            axes[i].axis('off')
    
    for i in range(len(available_steps), len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"📸 Timeline guardada: {save_path.name}")
    plt.close()

def extract_frames_from_video(video_path, num_frames=1, start_sec=3.0):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    start_frame = int(start_sec * fps)
    if start_frame >= total_frames:
        cap.release()
        return []
    
    frame_indices = np.linspace(start_frame, total_frames-1, num_frames, dtype=int)
    extracted_frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            extracted_frames.append(frame)
    cap.release()
    return extracted_frames

def create_debug_for_videos(max_videos_per_category=5):
    """Función principal de depuración"""
    if not TEMP_DIR.exists():
        print(f"❌ No existe TEMP_DIR: {TEMP_DIR}")
        return
    
    DEBUG_BASE.mkdir(parents=True, exist_ok=True)
    print(f"📁 Guardando debug en: {DEBUG_BASE}")
    
    for category in CATEGORIES:
        category_dir = TEMP_DIR / category
        if not category_dir.exists():
            print(f"⚠️ Carpeta no encontrada: {category}")
            continue
        
        debug_category = DEBUG_BASE / category
        debug_category.mkdir(parents=True, exist_ok=True)
        
        video_extensions = ['.mp4', '.MP4', '.avi', '.AVI', '.mov', '.MOV']
        videos = []
        for ext in video_extensions:
            videos.extend(category_dir.glob(f"*{ext}"))
        
        print(f"\n📂 Procesando categoría: {category.upper()} → {len(videos)} videos")
        
        for video_path in tqdm(videos[:max_videos_per_category], desc=f"Categoría {category}"):
            video_id = video_path.stem
            print(f"\n🎬 Procesando: {video_path.name}")
            
            frames = extract_frames_from_video(video_path, num_frames=1, start_sec=START_SEC)
            if not frames:
                print("   ⚠️ No se pudo extraer frame")
                continue
            
            frame = frames[0]
            final, steps = improved_palm_crop_adaptive(frame, debug=True)
            
            if final is None:
                print("   ⚠️ Falló el procesamiento")
                continue
            
            # 1. Guardar timeline (alta resolución)
            timeline_path = debug_category / f"{video_id}_timeline_steps.png"
            show_timeline_enhanced(steps, video_id, timeline_path)
            
            # 2. Guardar imágenes individuales clave (para artículo)
            key_steps = ['original', 'enhanced', 'skin_mask', 'morphology', 
                        'mask_overlay', 'center_detected', 'crop_box', 'final_result']
            
            for step_name in key_steps:
                if step_name in steps and steps[step_name] is not None:
                    img = steps[step_name]
                    if len(img.shape) == 3 and img.shape[2] == 3:
                        save_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if step_name != 'final_result' else img
                    else:
                        save_img = img
                    
                    out_path = debug_category / f"{video_id}_{step_name}.png"
                    if step_name == 'final_result':
                        out_path = debug_category / f"{video_id}_final.jpg"
                        cv2.imwrite(str(out_path), img)
                    else:
                        cv2.imwrite(str(out_path), save_img if len(save_img.shape)==2 else cv2.cvtColor(save_img, cv2.COLOR_RGB2BGR))
            
            print(f"   ✅ Debug completo para {video_id}")
    
    print("\n" + "="*80)
    print("🎉 ¡Depuración finalizada!")
    print(f"📁 Revisa la carpeta: {DEBUG_BASE}")
    print("Las imágenes están listas para tu artículo científico (alta resolución).")

# ====================== EJECUCIÓN ======================
if __name__ == "__main__":
    create_debug_for_videos(max_videos_per_category=5)  # Cambia este número según necesites

📁 Guardando debug en: C:\Users\Usuario\Desktop\UNIR\DATASETS\DEBUG_PREPROCESS

📂 Procesando categoría: NORMAL → 18 videos


Categoría normal:   0%|                                                                          | 0/5 [00:00<?, ?it/s]


🎬 Procesando: ID001.mp4
📸 Timeline guardada: ID001_timeline_steps.png


Categoría normal:  20%|█████████████▏                                                    | 1/5 [00:16<01:07, 16.84s/it]

   ✅ Debug completo para ID001

🎬 Procesando: ID003.mp4
📸 Timeline guardada: ID003_timeline_steps.png


...

   ✅ Debug completo para ID008

📂 Procesando categoría: LEVE → 0 videos


Categoría leve: 0it [00:00, ?it/s]



📂 Procesando categoría: MODERADA → 0 videos


Categoría moderada: 0it [00:00, ?it/s]


🎉 ¡Depuración finalizada!
📁 Revisa la carpeta: C:\Users\Usuario\Desktop\UNIR\DATASETS\DEBUG_PREPROCESS
Las imágenes están listas para tu artículo científico (alta resolución).
